In [ ]:
%pip install firebase_admin

In [ ]:
import os

import firebase_admin
from firebase_admin import credentials

# Point these at your own Firebase project (see the README).
cred = credentials.Certificate(os.environ["FIREBASE_CREDENTIALS"])  # path to a service account JSON key
firebase_admin.initialize_app(cred, {
    'databaseURL': os.environ["FIREBASE_DATABASE_URL"],
    'storageBucket': os.environ["FIREBASE_STORAGE_BUCKET"]
})

In [ ]:
import os, shutil

folder = 'data'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))

In [ ]:
import re

def extract_file(url):

    # Define the pattern to extract the desired substring
    pattern = r'elements%2F(.*?.png)'

    # Use regex to find the matching substring
    match = re.search(pattern, url)

    if match:
        extracted_string = match.group(1)
        return extracted_string
    else:
        print("No match found.")
        
print(file)

In [ ]:
from firebase_admin import db, storage

bucket = storage.bucket()
print(bucket)
# for key in bucket.keys():
#     print(f"Bucket key: {key}")

ref = db.reference("/")
all_dates = ref.get()
print(all_dates.keys())
for date in all_dates.keys():
    if date == "stats":
        # ignore our stats counter
        continue
    all_drawings = ref.child(date).get()
    for drawing in all_drawings.keys():
        
        elements = ref.child(date).child(drawing).child("elements").get()
        print(f"DRAWING: {drawing}")
        print()
        print("Iterating elements:")
        for element in elements:
            print()
            if element == None:
                continue
            print(element['type'])
            if element['type'] == "unknown":
                continue
            print(element)
            path = 'data/' + element['type'].lower()
            if not os.path.exists(path):
                os.mkdir(path)
            url = element['url']
            print(url)
            print(extract_file(url))
            blob = bucket.blob(date + '/' + drawing + '/elements/' + extract_file(url))
            print(blob)
            print(blob.exists())
            blob.download_to_filename(path + "/" + extract_file(url))
        
